# 00 — Pipeline complet sur Colab (données → baseline → LoRA → évaluation)
Tout ce qui est lourd est stocké sur **Google Drive** (`MyDrive/lora-zellige/`) : on peut reprendre après une déconnexion.
Exécution > Modifier le type d'exécution > **GPU T4**. Lancer les cellules dans l'ordre ; chaque étape déjà faite est sautée.

In [ ]:
# 1. Setup : Drive + code + dépendances + liens vers Drive
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
D = '/content/drive/MyDrive/lora-zellige'
os.chdir('/content')
if not os.path.exists('lora-zellige'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/braziyass/lora-zellige.git'], check=True)
os.chdir('/content/lora-zellige')
subprocess.run(['git', 'pull', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
for sub in ['data/raw', 'data/processed', 'outputs', 'results/generations']:
    os.makedirs(f'{D}/{sub}', exist_ok=True)
    if not os.path.islink(sub):
        subprocess.run(['rm', '-rf', sub]); os.makedirs(os.path.dirname(sub), exist_ok=True)
        os.symlink(f'{D}/{sub}', sub)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN — activez le GPU T4')
print('Images brutes sur Drive :', len(os.listdir('data/raw')))

In [ ]:
# 2. Téléchargement Wikimedia Commons (licences libres uniquement)
SRC = f'{D}/sources_downloaded.csv'
for cat, n in [('Zellige', 60), ('Zellige in Morocco', 30), ('Zellige in Fez', 30), ('Zellige in Marrakesh', 30)]:
    subprocess.run([sys.executable, 'data/download_commons.py', '--category', cat, '--limit', str(n), '--sources', SRC])
import pandas as pd
src = pd.read_csv(SRC); print(len(src), 'images'); print(src['license'].value_counts())

In [ ]:
# 3. Planches contact numérotées (pour choisir et légender)
import matplotlib.pyplot as plt
from PIL import Image
src = pd.read_csv(f'{D}/sources_downloaded.csv')
per = 12
for start in range(0, len(src), per):
    chunk = src.iloc[start:start+per]
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for ax in axes.flat: ax.axis('off')
    for ax, (i, r) in zip(axes.flat, chunk.iterrows()):
        try: ax.imshow(Image.open(f"data/raw/{r['file']}").convert('RGB'))
        except Exception as e: ax.set_title(str(e)[:30])
        ax.set_title(f"#{i}  {r['file']}", fontsize=10)
    plt.tight_layout(); plt.show()

In [ ]:
# 4. Infos texte des images (licence, description Commons)
pd.set_option('display.max_colwidth', 120); pd.set_option('display.max_rows', 300)
print(src[['file', 'license', 'author', 'commons_description']].to_csv(sep='|'))

In [ ]:
# 5. Préparation : utilise data/sources.csv (légendes) du dépôt
subprocess.run(['git', 'pull', '-q'], check=True)
!python -m src.prepare_data --config configs/base.yaml

In [ ]:
# 6. Baseline (modèle de base seul)
!python -m src.generate --config configs/base.yaml --base_only

In [ ]:
# 7. Balayage des rangs 4/8/16/32 (reprend là où il s'est arrêté)
!python -m src.sweep --sweep configs/sweep.yaml

In [ ]:
# 8. Génération de tous les checkpoints + évaluation
!python -m src.generate --config configs/base.yaml
!python -m src.evaluate --config configs/base.yaml
!mkdir -p {D}/results && cp -r results/*.csv results/*.json results/figures {D}/results/ && ls {D}/results/figures